# Master Thesis Analysis: LLM Knowledge Cutoffs and Python Library Diffusion

**Author:** Roland Tuboly  
**Project:** Empirical analysis of whether LLM training cutoffs (e.g., September 2021) affect the diffusion of Python libraries released around the cutoff.

## 1. Research Context and Objective
This notebook implements a **Regression Discontinuity Design (RDD)** to study the diffusion of Python libraries around documented LLM knowledge cutoffs. We test the hypothesis that libraries released shortly before a cutoff—and thus potentially included in the model's training data—diffuse more widely than those released shortly after.

### Core Components:
- **Running Variable:** Weeks from the library's initial release to the LLM training cutoff.
- **Primary Cutoff:** September 27, 2021.
- **Methodology:** Local linear RDD (`rdrobust`) as the primary estimator, supplemented by clustered WLS and permutation-based inference.

## 2. Setup and Configuration
We first import necessary libraries and define shared parameters used throughout the analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from pathlib import Path
from scipy.stats import norm
import rdrobust
from IPython.display import Image, display

# Paths
BASE_DIR = Path(".").resolve()
RAW_DIR = BASE_DIR / "data" / "raw"
INTERM_DIR = BASE_DIR / "data" / "intermediate"
FINAL_DIR = BASE_DIR / "data" / "final"
RESULTS_DIR = BASE_DIR / "results"
FIGURES_DIR = RESULTS_DIR / "figures"

for d in [INTERM_DIR, FINAL_DIR, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Analysis Parameters
WINDOW_WEEKS = 52
DONUT_WEEKS = [-2, -1, 0, 1, 2]
HORIZON_WEEKS = 52
MAIN_CUTOFF_NAME = "Main_2021"
MAIN_CUTOFF_DATE = pd.Timestamp("2021-09-27")

# Cutoff Environments for placebo/multi-cutoff tests
CUTOFFS = {
    "Placebo_2018": pd.Timestamp("2018-09-24"),
    "Placebo_2019": pd.Timestamp("2019-09-30"),
    "Placebo_2020": pd.Timestamp("2020-09-28"),
    MAIN_CUTOFF_NAME: MAIN_CUTOFF_DATE,
    "Adoption_2023": pd.Timestamp("2023-01-02")
}

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

### Utility Functions
Consistent variable definitions and estimation wrappers are crucial for reproducibility.

In [ ]:
def normalize_name(s: pd.Series) -> pd.Series:
    """Normalize package/library names for consistent merging."""
    return (
        s.astype(str)
        .str.lower()
        .str.replace("_", "-", regex=False)
        .str.strip()
    )

def check_monday_alignment(df: pd.DataFrame, date_col: str = "week_start") -> None:
    """Assert that all date values in a column are Monday-aligned."""
    bad_weekdays = df.loc[df[date_col].dt.weekday != 0, date_col].drop_duplicates()
    if not bad_weekdays.empty:
        raise ValueError(f"Found non-Monday values in {date_col}: {bad_weekdays.head().tolist()}")

def get_weeks_since(start_col: pd.Series, end_col: pd.Series) -> pd.Series:
    """Calculate the number of weeks between two dates."""
    return ((start_col - end_col).dt.days // 7).astype("int64")

def run_local_linear_rdd(df: pd.DataFrame, outcome_col: str, h: float = None, donut_weeks: list = None, label: str = "", cluster_col: str = None):
    """
    Implements a Local Linear RDD using Weighted Least Squares (WLS) with a triangular kernel.
    Specification: log(1+Y) ~ alpha + beta*Post + gamma1*Dist + gamma2*Dist*Post
    """
    sub = df.copy()
    if donut_weeks:
        sub = sub[~sub["dist_to_cutoff"].isin(donut_weeks)]
    
    sub = sub[(sub["dist_to_cutoff"] >= -h) & (sub["dist_to_cutoff"] <= h)].copy()
    sub = sub.dropna(subset=[outcome_col])
    
    if len(sub) < 10:
        return {"Label": label, "Outcome": outcome_col, "Estimate": np.nan, "Std.Err": np.nan, "P-value": np.nan, "N": len(sub), "BW": h, "Method": "WLS"}

    if outcome_col.startswith("cum") or "downloads" in outcome_col:
        sub["y"] = np.log1p(sub[outcome_col])
    else:
        sub["y"] = sub[outcome_col]

    sub["x"] = sub["dist_to_cutoff"]
    sub["treated"] = (sub["x"] >= 0).astype(int)
    sub["x_treated"] = sub["x"] * sub["treated"]
    sub["weight"] = 1 - (np.abs(sub["x"]) / h)
    sub = sub[sub["weight"] > 0]
    
    try:
        cov_type = 'HC1'
        fit_kwargs = {'cov_type': cov_type}
        if cluster_col and cluster_col in sub.columns:
            sub['cluster_idx'] = sub[cluster_col].astype('category').cat.codes
            fit_kwargs['cov_type'] = 'cluster'
            fit_kwargs['cov_kwds'] = {'groups': sub['cluster_idx']}

        model = smf.wls("y ~ treated + x + x_treated", data=sub, weights=sub["weight"]).fit(**fit_kwargs)
        return {"Label": label, "Outcome": outcome_col, "Estimate": model.params["treated"], "Std.Err": model.bse["treated"], "P-value": model.pvalues["treated"], "N": int(model.nobs), "BW": h, "Method": f"WLS ({fit_kwargs['cov_type']})"}
    except Exception as e:
        return {"Label": label, "Outcome": outcome_col, "Estimate": np.nan, "Std.Err": np.nan, "P-value": np.nan, "N": 0, "BW": h, "Method": "WLS"}

## Research Design: Parameter Selection & Rationale

### 1. Outcome Horizons (12, 26, 52 Weeks)
- **Rationale:** Allows for the study of **diffusion dynamics**—specifically whether potential effects are temporary delays or persistent exclusion. 52 weeks captures "long-run" diffusion.

### 2. Bandwidth (h = 26 Weeks)
- **Rationale:** Balances the **bias-variance tradeoff**.

### 3. Donut Hole Specification ([-2, -1, 0, 1, 2])
- **Rationale:** Accounts for **measurement error** in the release-week proxy and **conceptual ambiguity** near the cutoff.

### 4. Minimum Download Thresholds ([0, 10, 100])
- **Rationale:** Filters out "toy" libraries and abandoned projects, ensuring findings are robust across the adoption spectrum.

### 5. Identification Assumptions
- **Continuity:** We assume smooth potential adoption across the threshold in the absence of treatment.
- **No Manipulation:** Strategic timing of releases relative to the secret, retrospective cutoff is unlikely.
- **Seasonality:** Addressed via placebo years and Diff-in-RDD, though these remain supportive analyses.

## 3. Data Construction: PyPI and GitHub
We construct a library-level dataset using the first observed positive download week as the release date proxy.

In [ ]:
# 3.1 Build PyPI Base
pypi_path = RAW_DIR / "pypi_downloads.parquet"
df_pypi = pd.read_parquet(pypi_path, columns=["project", "week_start", "downloads"])
df_pypi["package"] = normalize_name(df_pypi["project"])
df_pypi["week_start"] = pd.to_datetime(df_pypi["week_start"])
df_pypi["downloads"] = pd.to_numeric(df_pypi["downloads"], errors="coerce").fillna(0).astype("int64")

# Release Proxy
valid_downloads = df_pypi.loc[df_pypi["downloads"] > 0, ["package", "week_start"]].copy()
release_dates = valid_downloads.groupby("package", as_index=False)["week_start"].min().rename(columns={"week_start": "release_week"})

# Aggregate PyPI Horizons
df_pypi = df_pypi.merge(release_dates, on="package", how="inner")
df_pypi["weeks_since_release"] = get_weeks_since(df_pypi["week_start"], df_pypi["release_week"])
df_pypi_post = df_pypi.loc[df_pypi["weeks_since_release"] >= 0].copy()

pypi_base = release_dates.copy()
for h in [12, 26, 52]:
    agg = df_pypi_post.loc[df_pypi_post["weeks_since_release"] < h].groupby("package")["downloads"].sum().reset_index()
    pypi_base = pypi_base.merge(agg.rename(columns={"downloads": f"cum_downloads_{h}wk"}), on="package", how="left").fillna(0)

pypi_base["total_downloads_52wk"] = pypi_base["cum_downloads_52wk"]

# 3.2 Aggregate GitHub
gh_path = RAW_DIR / "github_library_week_panel.csv"
df_gh = pd.read_csv(gh_path)
df_gh["package"] = normalize_name(df_gh["library"])
df_gh["week_start"] = pd.to_datetime(df_gh["week_start"])
df_gh["import_count"] = pd.to_numeric(df_gh["import_count"], errors="coerce").fillna(0)

df_gh = df_gh.merge(release_dates, on="package", how="inner")
df_gh["weeks_since_release"] = get_weeks_since(df_gh["week_start"], df_gh["release_week"])
df_gh_post = df_gh.loc[df_gh["weeks_since_release"] >= 0].copy()

gh_agg = pd.DataFrame({"package": pypi_base["package"].unique()})
for h in [12, 26, 52]:
    agg = df_gh_post.loc[df_gh_post["weeks_since_release"] < h].groupby("package")["import_count"].sum().reset_index()
    gh_agg = gh_agg.merge(agg.rename(columns={"import_count": f"cum_imports_{h}wk"}), on="package", how="left")

## 4. Merging and Sample Selection
We merge PyPI and GitHub data and apply temporal restrictions to ensure libraries have full outcome data.

In [ ]:
def prep_analysis_data(date, pypi_base, gh_agg):
    df = pypi_base.merge(gh_agg, on="package", how="left", indicator="gh_merge")
    df["matched_to_github"] = (df["gh_merge"] == "both").astype(int)
    df["dist_to_cutoff"] = get_weeks_since(df["release_week"], date)
    df["in_donut"] = df["dist_to_cutoff"].isin(DONUT_WEEKS).astype(int)
    
    # Time window restriction
    df = df.loc[(df["dist_to_cutoff"].abs() <= WINDOW_WEEKS)].copy()
    
    # Data availability check
    pypi_max = df_pypi["week_start"].max()
    df["last_needed"] = df["release_week"] + pd.to_timedelta(51 * 7, unit="D")
    df = df.loc[df["last_needed"] <= pypi_max].copy()
    
    return df

df_main = prep_analysis_data(MAIN_CUTOFF_DATE, pypi_base, gh_agg)
print(f"Sample Size (Main 2021): {len(df_main):,}")

## 5. Diagnostics (Identification Checks)
We verify the continuity of the running variable and inspect the binned scatterplots.

In [ ]:
# 5.1 Density Plot
counts = df_main.groupby("dist_to_cutoff").size()
plt.bar(counts.index, counts.values, color="steelblue", alpha=0.7)
plt.axvline(0, color="red", linestyle="--", label="Cutoff")
plt.title("Density of Python Library Releases")
plt.xlabel("Weeks from Cutoff")
plt.show()

# 5.2 Binned Scatterplot
df_main["log_downloads"] = np.log1p(df_main["total_downloads_52wk"])
binscatter = df_main.groupby("dist_to_cutoff")["log_downloads"].mean().reset_index()
sns.scatterplot(data=binscatter, x="dist_to_cutoff", y="log_downloads")
plt.axvline(0, color="red", linestyle="--")
plt.title("Binned Scatter: Log Downloads by Release Week")
plt.show()

## 6. Main RDD Results
We estimate the treatment effect using primary and secondary estimators. Our baseline results for 2021 are statistically null, suggesting no broad 'Knowledge Wall' effect.

In [ ]:
results = []
for thresh in [0, 10, 100]:
    sub = df_main[df_main["total_downloads_52wk"] >= thresh]
    res = run_local_linear_rdd(sub, "total_downloads_52wk", h=26, donut_weeks=DONUT_WEEKS, label=f"Min {thresh} DL", cluster_col="dist_to_cutoff")
    results.append(res)

res_df = pd.DataFrame(results)
print("RDD Results (Baseline Specifications):")
print(res_df.round(4))

## 10. Advanced Analysis: Catch-up and Distributional Effects

We extend the analysis to test for **diffusion catch-up** and **distributional sensitivity** using Quantile RDD. These tests investigate if the LLM cutoff creates temporary adoption delays rather than permanent exclusion.

### 10.1 Catch-up Dynamics (Horizon Comparison)
Comparing RDD estimates at 12, 26, and 52 weeks suggests a potential recovery from initial adoption 'taxes,' though estimates are noisy.

In [ ]:
horizon_plot_path = FIGURES_DIR / "rdd_horizon_coefficients.png"
if horizon_plot_path.exists():
    display(Image(filename=str(horizon_plot_path)))
    print("Note the suggestive recovery pattern across horizons.")

### 10.2 Distributional Sensitivity (Quantile RDD)
Quantile RDD (e.g., Median) results can show a 'Post-Cutoff Bonus' or 'Freshness Premium,' though this finding is sensitive to the choice of estimator and baseline.

In [ ]:
quantile_plot_path = FIGURES_DIR / "rdd_quantile_coefficients.png"
if quantile_plot_path.exists():
    display(Image(filename=str(quantile_plot_path)))

## 11. Historical Context: The "Relative Suppression" Framework

To determine if the 2021 result is unusual, we compare it to a stacked average of historical placebo cutoffs. This analysis suggests that 2021 adoption momentum was lower than historical seasonal averages, though this interpretation depends on the choice of seasonal baseline.

In [ ]:
stacked_res_path = RESULTS_DIR / "stacked_rdd_results.csv"
if stacked_res_path.exists():
    stacked_df = pd.read_csv(stacked_res_path)
    display(stacked_df[["Label", "Estimate", "Std.Err", "P-value", "N"]].round(4))

## 12. Conclusion and Synthesis

The empirical evidence suggests a resilient software ecosystem:
1. **Main Result:** We find no robust evidence of a broad LLM 'knowledge wall' for Python libraries in 2021.
2. **Seasonality:** Adoption patterns are heavily influenced by seasonal cycles, which complicate naive cutoff comparisons.
3. **Suggestive Catch-up:** Any early adoption delays appear temporary, with successful libraries overcoming potential knowledge gaps within their first year.